---
## ⚙️ Instalação das dependências

In [23]:
# Instalação das dependências do Projeto

!sudo apt-get update -qq && sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q dspy ollama requests ipywidgets

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [24]:
import subprocess
import threading
import time
import requests

model_llm = "gemma3:4b"

print("Tentando parar qualquer instância existente do Ollama server...")
!pkill -f "ollama serve" || true
time.sleep(1)

print("Iniciando Ollama server em background...")
!nohup ollama serve > ollama_server.log 2>&1 &

print("Aguardando servidor...", end="")

for _ in range(20):
    try:
        r = requests.get("http://localhost:11434/", timeout=2)
        if r.status_code == 200:
            print(" ✅ Online!")
            break
    except Exception:
        pass
    time.sleep(2)
    print(".", end="", flush=True)
else:
    print("Timeout: Ollama server não iniciou ou não está acessível.")

print(f"Subindo modelo {model_llm}")
!ollama pull {model_llm}
print(f"Modelo pronto!")

Tentando parar qualquer instância existente do Ollama server...
^C
Iniciando Ollama server em background...
Aguardando servidor.... ✅ Online!
Subindo modelo gemma3:4b

Modelo pronto!


In [25]:
import os
from google.colab import files

DB_PATH = "artistas-pop.db"

if not os.path.exists(DB_PATH):
    print(f"Escolha um schema do banco '{DB_PATH}'")
    uploaded = files.upload()
    if DB_PATH not in uploaded:
        raise FileNotFoundError(
            f"Arquivo '{DB_PATH}' não encontrado. "
            "Certifique-se de fazer upload com esse nome exato."
        )
    print(f"✅ '{DB_PATH}' carregado com sucesso!")
else:
    print(f"✅ '{DB_PATH}' já está disponível.")

✅ 'artistas-pop.db' já está disponível.


In [26]:
import sqlite3
import json
from textwrap import dedent
from typing import Optional

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
conn.row_factory = sqlite3.Row
print(f"Conectado a '{DB_PATH}'")


def executar_sql(sql: str, max_rows: int = 50):
    """Executa um SQL e retorna (colunas, linhas, erro)."""
    try:
        sql_limpo = sql.strip()
        for tag in ["```sql", "```sqlite", "```"]:
            sql_limpo = sql_limpo.replace(tag, "")
        sql_limpo = sql_limpo.strip().rstrip(";")

        cur = conn.execute(sql_limpo)
        rows = cur.fetchmany(max_rows)
        colunas = [d[0] for d in cur.description] if cur.description else []
        return colunas, [dict(zip(colunas, r)) for r in rows], None
    except Exception as e:
        return [], [], str(e)


def resumo_bd():
    """Retorna um snapshot rápido do banco para diagnóstico."""

    _, rows, erro = executar_sql("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
    """)

    if erro:
      print("Erro:", erro)
      return

    tabelas = [r["name"] for r in rows] if rows else []
    print("Tabelas encontradas:", tabelas)

    for tabela in tabelas:
        _, resultado, erro = executar_sql(
            f"SELECT COUNT(*) AS total FROM {tabela}"
        )

        if erro:
            print(f"{tabela}: erro ao contar registros")
            continue

        print(f"  {tabela}: {resultado[0]['total']} linhas")

resumo_bd()

Conectado a 'artistas-pop.db'
Tabelas encontradas: ['albuns', 'artistas', 'charts']
  albuns: 17 linhas
  artistas: 8 linhas
  charts: 17 linhas


In [27]:
import dspy

lm = dspy.LM(
    model=f"ollama/{model_llm}",
    api_base="http://localhost:11434",
    max_tokens=600,
    temperature=0.0,
    cache=False,
    trace=True,
)
dspy.configure(lm=lm)
print(f"DSPy configurado com sucesso")


from textwrap import dedent

SCHEMA = dedent("""
    Banco SQLite: artistasPop-schema.db  (dados de artistas musicais, álbuns e desempenho em charts)

    Tabela artistas
    - id INTEGER
    - name TEXT
    - debut_year INTEGER
    - genre TEXT

    Tabela albuns
    - id INTEGER
    - artist_id INTEGER
    - title TEXT
    - release_year INTEGER
    - sales_millions REAL

    Tabela charts
    - id INTEGER
    - album_id INTEGER
    - peak_position INTEGER
    - weeks_on_chart INTEGER

    RELACIONAMENTOS:

      artistas (1) ────── (N) albuns
      albuns   (1) ────── (N) charts

    REGRAS IMPORTANTES:

      1. Sempre use JOIN entre albuns e artistas via artist_id.
      2. Sempre use JOIN entre charts e albuns via album_id quando precisar
         de informações do artista ou do álbum.
      3. Menor peak_position representa melhor desempenho no chart.
         Exemplo: posição 1 é melhor que posição 2.
      4. Maior weeks_on_chart representa maior permanência na parada.
      5. Maior sales_millions representa maior volume de vendas.
      6. Para encontrar os artistas mais vendidos, agregue sales_millions
         dos álbuns do artista.
      10. Nunca use SELECT * — liste explicitamente as colunas necessárias.
      11. Retorne apenas o SQL, sem explicações.
""").strip()


class GerarSQL(dspy.Signature):
    """Você é um especialista em SQLite. Com base no schema fornecido, gere
    exatamente um comando SELECT válido para responder à pergunta do usuário.
    Retorne APENAS o SQL, sem markdown, sem explicações."""

    schema: str     = dspy.InputField(desc="Schema do banco")
    question: str   = dspy.InputField(desc="Pergunta em linguagem natural (pode ser em português ou inglês)")
    sql_query: str  = dspy.OutputField(desc="Comando SELECT SQLite válido, sem markdown")


class RefinarSQL(dspy.Signature):
    """Corrija o SQL com base no erro retornado pelo banco.
    Retorne APENAS o SQL corrigido, sem markdown."""

    schema: str     = dspy.InputField(desc="Schema do banco")
    question: str   = dspy.InputField(desc="Pergunta original")
    sql_query: str  = dspy.InputField(desc="SQL com problema")
    error: str      = dspy.InputField(desc="Mensagem de erro SQLite")
    sql_query: str  = dspy.OutputField(desc="SQL corrigido")


class GerarResposta(dspy.Signature):
    """Formule uma resposta clara e completa em português brasileiro para o usuário.
    Interprete os dados retornados pelo banco, destaque os valores mais relevantes
    e, quando possível, forneça contexto ou insight sobre os números."""

    question: str       = dspy.InputField(desc="Pergunta original do usuário")
    sql_query: str      = dspy.InputField(desc="SQL que foi executado")
    data: str           = dspy.InputField(desc="Dados retornados pelo banco (JSON)")
    answer: str         = dspy.OutputField(desc="Resposta em português, clara e informativa")


print("✅ Assinaturas DSPy definidas")

DSPy configurado com sucesso
✅ Assinaturas DSPy definidas


/usr/local/lib/python3.12/dist-packages/dspy/signatures/signature.py:185: UserWarning: Field name "schema" in "GerarSQL" shadows an attribute in parent "Signature"
  cls = super().__new__(mcs, signature_name, bases, namespace, **kwargs)
/usr/local/lib/python3.12/dist-packages/dspy/signatures/signature.py:185: UserWarning: Field name "schema" in "RefinarSQL" shadows an attribute in parent "Signature"
  cls = super().__new__(mcs, signature_name, bases, namespace, **kwargs)


In [28]:
EXEMPLOS_SQL = [

    # ── Artistas ────────────────────────────────────────────────────────────

    dspy.Example(
        schema=SCHEMA,
        question="Quais artistas existem na base de dados?",
        sql_query=(
            "SELECT id, name, genre, debut_year "
            "FROM artistas "
            "ORDER BY name"
        )
    ).with_inputs("schema", "question"),

    dspy.Example(
        schema=SCHEMA,
        question="Quais artistas do gênero Pop existem na base?",
        sql_query=(
            "SELECT id, name, debut_year "
            "FROM artistas "
            "WHERE genre LIKE '%Pop%' "
            "ORDER BY name"
        )
    ).with_inputs("schema", "question"),

    # ── Álbuns por artista ──────────────────────────────────────────────────

    dspy.Example(
        schema=SCHEMA,
        question="Quais álbuns Lady Gaga lançou?",
        sql_query=(
            "SELECT a.title, a.release_year, a.sales_millions "
            "FROM albuns a "
            "JOIN artistas ar ON ar.id = a.artist_id "
            "WHERE ar.name = 'Lady Gaga' "
            "ORDER BY a.release_year"
        )
    ).with_inputs("schema", "question"),

    dspy.Example(
        schema=SCHEMA,
        question="Quais álbuns Beyonce lançou?",
        sql_query=(
            "SELECT a.title, a.release_year, a.sales_millions "
            "FROM albuns a "
            "JOIN artistas ar ON ar.id = a.artist_id "
            "WHERE ar.name = 'Beyonce' "
            "ORDER BY a.release_year"
        )
    ).with_inputs("schema", "question"),

    # ── Vendas ──────────────────────────────────────────────────────────────

    dspy.Example(
        schema=SCHEMA,
        question="Qual o álbum mais vendido da base?",
        sql_query=(
            "SELECT title, sales_millions "
            "FROM albuns "
            "ORDER BY sales_millions DESC "
            "LIMIT 1"
        )
    ).with_inputs("schema", "question"),

    dspy.Example(
        schema=SCHEMA,
        question="Quais são os 5 álbuns mais vendidos?",
        sql_query=(
            "SELECT title, sales_millions "
            "FROM albuns "
            "ORDER BY sales_millions DESC "
            "LIMIT 5"
        )
    ).with_inputs("schema", "question"),

    dspy.Example(
        schema=SCHEMA,
        question="Qual artista possui mais vendas acumuladas?",
        sql_query=(
            "SELECT ar.name, "
            "SUM(a.sales_millions) AS total_vendas "
            "FROM artistas ar "
            "JOIN albuns a ON a.artist_id = ar.id "
            "GROUP BY ar.id, ar.name "
            "ORDER BY total_vendas DESC "
            "LIMIT 1"
        )
    ).with_inputs("schema", "question"),

    dspy.Example(
        schema=SCHEMA,
        question="Mostre as vendas totais por artista.",
        sql_query=(
            "SELECT ar.name, "
            "SUM(a.sales_millions) AS total_vendas "
            "FROM artistas ar "
            "JOIN albuns a ON a.artist_id = ar.id "
            "GROUP BY ar.id, ar.name "
            "ORDER BY total_vendas DESC"
        )
    ).with_inputs("schema", "question"),

    # ── Charts ──────────────────────────────────────────────────────────────

    dspy.Example(
        schema=SCHEMA,
        question="Quais álbuns chegaram ao primeiro lugar do chart?",
        sql_query=(
            "SELECT a.title, ar.name "
            "FROM charts c "
            "JOIN albuns a ON a.id = c.album_id "
            "JOIN artistas ar ON ar.id = a.artist_id "
            "WHERE c.peak_position = 1 "
            "ORDER BY a.title"
        )
    ).with_inputs("schema", "question"),

    dspy.Example(
        schema=SCHEMA,
        question="Qual álbum permaneceu mais tempo nas paradas?",
        sql_query=(
            "SELECT a.title, ar.name, c.weeks_on_chart "
            "FROM charts c "
            "JOIN albuns a ON a.id = c.album_id "
            "JOIN artistas ar ON ar.id = a.artist_id "
            "ORDER BY c.weeks_on_chart DESC "
            "LIMIT 1"
        )
    ).with_inputs("schema", "question"),

    dspy.Example(
        schema=SCHEMA,
        question="Quais são os 5 álbuns com mais semanas em chart?",
        sql_query=(
            "SELECT a.title, ar.name, c.weeks_on_chart "
            "FROM charts c "
            "JOIN albuns a ON a.id = c.album_id "
            "JOIN artistas ar ON ar.id = a.artist_id "
            "ORDER BY c.weeks_on_chart DESC "
            "LIMIT 5"
        )
    ).with_inputs("schema", "question"),

    # ── Comparações ─────────────────────────────────────────────────────────

    dspy.Example(
        schema=SCHEMA,
        question="Compare as vendas de todos os álbuns da Madonna.",
        sql_query=(
            "SELECT a.title, a.sales_millions "
            "FROM albuns a "
            "JOIN artistas ar ON ar.id = a.artist_id "
            "WHERE ar.name = 'Madonna' "
            "ORDER BY a.sales_millions DESC"
        )
    ).with_inputs("schema", "question"),

    dspy.Example(
        schema=SCHEMA,
        question="Qual artista possui mais álbuns cadastrados?",
        sql_query=(
            "SELECT ar.name, COUNT(a.id) AS quantidade_albuns "
            "FROM artistas ar "
            "JOIN albuns a ON a.artist_id = ar.id "
            "GROUP BY ar.id, ar.name "
            "ORDER BY quantidade_albuns DESC "
            "LIMIT 1"
        )
    ).with_inputs("schema", "question"),

    dspy.Example(
        schema=SCHEMA,
        question="Quantos álbuns existem por gênero musical?",
        sql_query=(
            "SELECT ar.genre, COUNT(a.id) AS quantidade_albuns "
            "FROM artistas ar "
            "JOIN albuns a ON a.artist_id = ar.id "
            "GROUP BY ar.genre "
            "ORDER BY quantidade_albuns DESC"
        )
    ).with_inputs("schema", "question"),

    # ── Períodos ────────────────────────────────────────────────────────────

    dspy.Example(
        schema=SCHEMA,
        question="Quais álbuns foram lançados após 2015?",
        sql_query=(
            "SELECT title, release_year "
            "FROM albuns "
            "WHERE release_year > 2015 "
            "ORDER BY release_year"
        )
    ).with_inputs("schema", "question"),

    dspy.Example(
        schema=SCHEMA,
        question="Qual foi o álbum mais vendido da década de 2010?",
        sql_query=(
            "SELECT title, sales_millions "
            "FROM albuns "
            "WHERE release_year BETWEEN 2010 AND 2019 "
            "ORDER BY sales_millions DESC "
            "LIMIT 1"
        )
    ).with_inputs("schema", "question"),

    # ── Estatísticas ────────────────────────────────────────────────────────

    dspy.Example(
        schema=SCHEMA,
        question="Qual a média de vendas dos álbuns cadastrados?",
        sql_query=(
            "SELECT AVG(sales_millions) AS media_vendas "
            "FROM albuns"
        )
    ).with_inputs("schema", "question"),

    dspy.Example(
        schema=SCHEMA,
        question="Qual a média de semanas em chart?",
        sql_query=(
            "SELECT AVG(weeks_on_chart) AS media_semanas "
            "FROM charts"
        )
    ).with_inputs("schema", "question"),
]

print(f"✅ {len(EXEMPLOS_SQL)} exemplos de treinamento carregados")

✅ 18 exemplos de treinamento carregados


In [29]:
class ArtistaQA(dspy.Module):
    """
    Pipeline completo:
      1. Gera SQL a partir de pergunta em linguagem natural
      2. Executa no banco SQLite
      3. Refina o SQL se houver erro (até 2 tentativas)
      4. Converte os resultados em resposta em português
    """

    def __init__(self, exemplos: list):
        super().__init__()
        self.gerador = dspy.Predict(GerarSQL)
        self.refinador = dspy.ChainOfThought(RefinarSQL)
        self.respondedor = dspy.ChainOfThought(GerarResposta)
        self.gerador.demos = exemplos

    def _limpar_sql(self, texto: str) -> str:
        """Remove markdown e espaços extras do SQL gerado."""
        for tag in ["```sql", "```sqlite", "```"]:
            texto = texto.replace(tag, "")

        linhas = [l.strip() for l in texto.strip().splitlines() if l.strip()]
        sql_lines = []
        capturando = False
        for linha in linhas:
            if linha.upper().startswith("SELECT"):
                capturando = True
            if capturando:
                sql_lines.append(linha)
        return " ".join(sql_lines).rstrip(";")

    def forward(self, question: str, verbose: bool = False):
        out = self.gerador(schema=SCHEMA, question=question)
        sql = self._limpar_sql(out.sql_query)

        if verbose:
            print(f"\n🔍 SQL gerado:\n{sql}\n")

        colunas, rows, erro = executar_sql(sql)

        tentativas = 0
        while erro and tentativas < 2:
            tentativas += 1
            if verbose:
                print(f"⚠️  Erro (tentativa {tentativas}): {erro}")
            refin = self.refinador(
                schema=SCHEMA,
                question=question,
                sql_query=sql,
                error=erro,
            )
            sql = self._limpar_sql(refin.sql_query)
            if verbose:
                print(f"🔧 SQL refinado (tentativa {tentativas}):\n{sql}\n")
            colunas, rows, erro = executar_sql(sql)

        if erro:
            return dspy.Prediction(
                sql_query=sql,
                data=[],
                answer=f"❌ Não foi possível executar a consulta após {tentativas} tentativas.\nErro: {erro}",
                error=erro,
            )

        dados_json = json.dumps(rows[:20], ensure_ascii=False, default=str)
        resp = self.respondedor(
            question=question,
            sql_query=sql,
            data=dados_json,
        )

        return dspy.Prediction(
            sql_query=sql,
            data=rows,
            answer=resp.answer,
            error=None,
        )


agente = ArtistaQA(exemplos=EXEMPLOS_SQL[:2])
print("✅ Agente pronto!")

✅ Agente pronto!


/usr/local/lib/python3.12/dist-packages/dspy/signatures/signature.py:185: UserWarning: Field name "schema" in "StringSignature" shadows an attribute in parent "Signature"
  cls = super().__new__(mcs, signature_name, bases, namespace, **kwargs)


In [30]:
PERGUNTAS_TESTE = [
    "Quais artistas existem na base de dados?",
    "Quais álbuns Lady Gaga lançou?",
    "Qual é o álbum mais vendido da base?",
    "Quais são os 5 álbuns mais vendidos?",
    "Qual artista possui mais vendas acumuladas?",
    "Quais álbuns chegaram ao primeiro lugar do chart?",
    "Qual álbum permaneceu mais tempo nas paradas?",
    "Quais são os 5 álbuns com mais semanas em chart?",
    "Qual artista possui mais álbuns cadastrados?",
    "Quais álbuns foram lançados após 2015?",
    "Qual foi o álbum mais vendido da década de 2010?",
    "Qual a média de vendas dos álbuns cadastrados?",
]

SEPARATOR = "─" * 70

for i, pergunta in enumerate(PERGUNTAS_TESTE, 1):
    print(f"\n{SEPARATOR}")
    print(f"❓ [{i}/{len(PERGUNTAS_TESTE)}] {pergunta}")
    print(SEPARATOR)

    resultado = agente(question=pergunta, verbose=True)

    print(f"📊 SQL: {resultado.sql_query}")
    print(f"📋 Linhas retornadas: {len(resultado.data)}")
    print(f"\n💬 Resposta:\n{resultado.answer}")

print(f"\n{SEPARATOR}")
print("✅ Testes concluídos!")


──────────────────────────────────────────────────────────────────────
❓ [1/12] Quais artistas existem na base de dados?
──────────────────────────────────────────────────────────────────────

🔍 SQL gerado:
SELECT id, name, genre, debut_year FROM artistas ORDER BY name

📊 SQL: SELECT id, name, genre, debut_year FROM artistas ORDER BY name
📋 Linhas retornadas: 8

💬 Resposta:
Os artistas na base de dados são: Beyonce, Billie Eilish, Charli XCX, Frank Ocean, Kendrick Lamar, Lady Gaga, Lana Del Rey e Madonna.

──────────────────────────────────────────────────────────────────────
❓ [2/12] Quais álbuns Lady Gaga lançou?
──────────────────────────────────────────────────────────────────────

🔍 SQL gerado:
SELECT albuns.title, albuns.release_year FROM albuns JOIN artistas ON albuns.artist_id = artistas.id WHERE artistas.name = 'Lady Gaga'

📊 SQL: SELECT albuns.title, albuns.release_year FROM albuns JOIN artistas ON albuns.artist_id = artistas.id WHERE artistas.name = 'Lady Gaga'
📋 Linhas ret

In [31]:
from IPython.display import display, HTML
import ipywidgets as widgets

input_box = widgets.Text(
    placeholder="Digite sua pergunta...",
    layout=widgets.Layout(width="85%"),
)
btn_enviar = widgets.Button(
    description="Enviar",
    button_style="primary",
    layout=widgets.Layout(width="12%"),
)
btn_limpar = widgets.Button(
    description="Limpar",
    button_style="warning",
    layout=widgets.Layout(width="10%"),
)
output = widgets.Output()

def ao_enviar(_):
    pergunta = input_box.value.strip()
    if not pergunta:
        return
    input_box.value = ""

    with output:
        display(HTML("<i>⏳ Processando...</i>"))

    resultado = agente(question=pergunta, verbose=False)

    with output:
        output.clear_output()

        display(HTML(f"""
        <div style='padding:10px;border:1px solid #ddd;border-radius:8px'>
            <b>Pergunta:</b> {pergunta}<br><br>
            <b>SQL:</b>
            <pre>{resultado.sql_query}</pre>

            <b>Resposta:</b>
            <p>{resultado.answer}</p>
        </div>
        """))

def ao_limpar(_):
    output.clear_output()

btn_enviar.on_click(ao_enviar)
btn_limpar.on_click(ao_limpar)
input_box.on_submit(ao_enviar)

# Layout
display(HTML("""
<div style='background:linear-gradient(135deg,#1565C0,#0288D1);
     color:white;padding:16px 20px;border-radius:8px;margin-bottom:12px'>
  <h2 style='margin:0'>Artistas Pop — IA que fala sobre artistas pop</h2>
  <p style='margin:4px 0 0 0;opacity:0.9;font-size:14px'>
    Pergunte em português sobre qualquer artista, álbum ou música que a ia responderá baseado em nosso banco de dados.
  </p>
</div>
"""))

display(widgets.HBox([input_box, btn_enviar]))
display(output)

Output()